In [1]:
from Bio.SeqUtils import molecular_weight as calculate_molecular_weight

import sys
sys.path.insert(1, '../scripts/') # comment out in python script
from load_environmental_variables import *
from utils import *
from utils_2 import *
import build_mrna_expression_reactions as build_mrna

load environmental variables
The project root is: /Users/joycebaghdassarian/Documents/UCSD/Lewis_Lab/Projects/human_me/


../scripts/build_trna_expression_reactions.py:48 UserWarning: Mature tRNA sequence not in the expected length range (76<=L<=93)


In [2]:
def translate_protein_cytosolic(gene_info):    

    # peptide bond formation: https://d1j63owfs0b5j3.cloudfront.net/pop-quiz/answerImage/Amino-Acid-1-popquiz.png
    # tRNA amino acide release: https://rnajournal.cshlp.org/content/14/8/1526/F1.expansion.html

    rxn = {charged_trna_map[aa_code]: -aa_count for aa_code, aa_count in gene_info.amino_acid_counts.items()} # tRNA consumption
    rxn[modified_trna_transcript_c] = gene_info.L_protein
    rxn[h2o_c] = -gene_info.L_protein # release of peptide from tRNA, addition of -OH to uncharged tRNA
    rxn[h_c] = gene_info.L_protein # release of peptide from tRNA, addition of -OH to uncharged tRNA
    rxn[h2o_c] += gene_info.L_protein - 1 # peptide bond formation (hydrolysis)
    
    # gtp hydrolysis per aa added
    rxn[ntp_map_c['G']] = -gene_info.L_protein 
    rxn[h2o_c] -= gene_info.L_protein
    rxn[ndp_map_c['G']] = gene_info.L_protein
    rxn[pi_c] = gene_info.L_protein
    rxn[h_c] += gene_info.L_protein
    

    rxn_c = rxn.copy()
    unfolded_protein_c = make_protein_metabolite(id_ = gene_info.hgnc_id + '_unfolded', 
                amino_acid_counts = gene_info.amino_acid_counts, L_protein = gene_info.L_protein,
                compartment = 'c')
    rxn_c[unfolded_protein_c] = 1
    
    translation_elongation = cobra.Reaction(gene_info.hgnc_id + 'TRANSLATION_ELONGATIONc')
    translation_elongation.subsytem = 'Protein_Expression'
    translation_elongation.add_metabolites(rxn_c)

    translation_elongation.gene_reaction_rule = ' and '.join(translation_efs + ['ribosome']) # GPRs

    return translation_elongation, unfolded_protein_c

def fold_protein_cytosolic(gene_info, unfolded_protein_c):
    # extending proteostasis network in the future would be good
    # will need to make sure inputs to each compartment-specific reactions are at the correct folding stage
    # e.g., mitochondria currently takes unfolded protein, and in future we may want it to take a partially folded
    
    folded_protein_c = unfolded_protein_c.copy()
    folded_protein_c.id = folded_protein_c.id.replace('unfolded', 'folded')
    rxn = {unfolded_protein_c: -1, folded_protein_c: 1}
    protein_folding = cobra.Reaction(gene_info.hgnc_id + '_CYTOSOLIC_PROTEIN_FOLDING')
    protein_folding.subsytem = 'Protein_Expression'
    
    if gene_info.L_protein > 100: #chaperone assisted for larger proteins - https://www.nature.com/articles/nature10317
        rxn = hydrolyze_atp(rxn, n_atp = gene_info.L_protein*proteolysis_translocation_atp_cost, compartment = 'c')
        protein_folding.gene_reaction_rule = ' and '.join(HSP40_c + HSP70_c) # GPRs
    
    
    protein_folding.add_metabolites(rxn)

    
    
    return protein_folding, folded_protein_c

Ubiquitin expression

In [3]:
# UBC
ubc_psim = psim_me[psim_me['HGNC_ID'] == 'HGNC:12468'] # UBC
ubc_psim['Location'] = 'c'
ubc_info = gene_information(metabolic_model = human_model, hgnc_id = ubc_psim['HGNC_ID'].values.tolist()[0], 
                         premrna_seq=ubc_psim['PREMRNA_SEQ'].values.tolist()[0], 
                            mrna_seq=ubc_psim['MRNA_SEQ'].values.tolist()[0], 
                            protein_seq=ubc_psim['PROTEIN_SEQ'].values.tolist()[0],
                            polyA_length = round(ubc_psim['POLYA_LENGTH'].values.tolist()[0]))
ubc_info.get_final_locations(human_model, final_locations=['c'])
ubc_mrna_expression_reactions = build_mrna.mrna_expression(ubc_info)

# ubiquitin monomer
single_ubiquitin_sequence = ubc_info.protein_seq[:76]
monoub_aa_counts = {k: single_ubiquitin_sequence.count(k) for k in amino_acids}
L_monoub = len(single_ubiquitin_sequence)
n_ub_monomers = ubc_info.protein_seq.count(single_ubiquitin_sequence)
ub_c = make_protein_metabolite(id_ = 'ubiquitin_monomer', amino_acid_counts = monoub_aa_counts,
                               L_protein = L_monoub, compartment = 'c')

# monomerization from ubc polyub
# amino_acid_counts_ubc = {k: ubc_info.protein_seq.count(k) for k in amino_acids}
# L_ubc = len(ubc_info.protein_seq)

ubc_translation_reaction_cytosolic, ubc_c = translate_protein_cytosolic(ubc_info)

ubiquitin_monomerization_ubc = cobra.Reaction(ubc_info.hgnc_id + '_MONOMERIZATIONc')
ubiquitin_monomerization_ubc.subsytem = 'Protein_Expression'
rxn = {ubc_c:-1, ub_c: n_ub_monomers, seq_amino_acid_map_c[ubc_info.protein_seq[n_ub_monomers*76:]]: 1, 
      h2o_c: -n_ub_monomers}
ubiquitin_monomerization_ubc.add_metabolites(rxn)
ubiquitin_monomerization_ubc.gene_reaction_rule = USP5[0]

# UBB
ubb_psim = psim_me[psim_me['HGNC_ID'] == 'HGNC:12463'] # UBB
ubb_psim['Location'] = 'c'
ubb_info = gene_information(metabolic_model = human_model, hgnc_id = ubb_psim['HGNC_ID'].values.tolist()[0], 
                         premrna_seq=ubb_psim['PREMRNA_SEQ'].values.tolist()[0], 
                            mrna_seq=ubb_psim['MRNA_SEQ'].values.tolist()[0], 
                            protein_seq=ubb_psim['PROTEIN_SEQ'].values.tolist()[0],
                            polyA_length = round(ubb_psim['POLYA_LENGTH'].values.tolist()[0]))
ubb_info.get_final_locations(human_model, final_locations=['c'])
ubb_mrna_expression_reactions = build_mrna.mrna_expression(ubb_info)

# amino_acid_counts_ubb = {k: ubb_info.protein_seq.count(k) for k in amino_acids}
# L_ubb = len(ubb_info.protein_seq)

ubb_translation_reaction_cytosolic, ubb_c = translate_protein_cytosolic(ubb_info)

# monomerization from ubb polyub
n_ub_monomers = ubb_info.protein_seq.count(single_ubiquitin_sequence)
ubiquitin_monomerization_ubb = cobra.Reaction(ubb_info.hgnc_id + '_MONOMERIZATIONc')
ubiquitin_monomerization_ubb.subsytem = 'Protein_Expression'
rxn = {ubb_c:-1, ub_c: n_ub_monomers, seq_amino_acid_map_c[ubb_info.protein_seq[n_ub_monomers*76:]]: 1, 
      h2o_c: -n_ub_monomers}
ubiquitin_monomerization_ubb.add_metabolites(rxn)
ubiquitin_monomerization_ubc.gene_reaction_rule = USP5[0]

# breakdown of the polyubiquitin cleaved from proteins in ubiquitin-proteasome pathway
polyub_aa_counts = {aa_code: aa_count*n_ub for aa_code, aa_count in monoub_aa_counts.items()}
polyub_c = make_protein_metabolite(id_ = 'cleaved_polyubiquitin_moiety', amino_acid_counts = polyub_aa_counts,
                               L_protein = L_monoub*n_ub, compartment = 'c')
ubiquitin_monomerization_polyub = cobra.Reaction('POLYUBIQUITIN_MONOMERIZATIONc')
ubiquitin_monomerization_polyub.subsytem = 'Protein_Expression'
rxn = {polyub_c:-1, ub_c: n_ub, h2o_c: -(n_ub-1)}
ubiquitin_monomerization_polyub.add_metabolites(rxn)
ubiquitin_monomerization_polyub.gene_reaction_rule = USP5[0]

# nuclear import of ubiquitin
nuclear_import_ub_mono = cobra.Reaction('UBIQUITIN_MONOMER_IMPORTtn')
nuclear_import_ub_mono.subsytem = 'Protein_Expression'
ub_n = ub_c.copy()
ub_n.id, ub_n.compartment = ub_n.id.replace('[c]', '[n]'), 'n'
nuclear_import_ub_mono.add_metabolites({ub_n: 1, ub_c: -1})
nuclear_import_ub_mono.lower_bound = -1000

# nuclear export of polyubiquitin moiety
nuclear_export_ub_poly = cobra.Reaction('POLYUBIQUITIN_MOIETY_EXPORTtn')
nuclear_export_ub_poly.subsytem = 'Protein_Expression'
polyub_n = polyub_c.copy()
polyub_n.id, polyub_n.compartment = polyub_n.id.replace('[c]', '[n]'), 'n'
nuclear_export_ub_poly.add_metabolites({polyub_n: -1, polyub_c: 1})
nuclear_export_ub_poly.lower_bound = -1000

# degradation
degradation_ub = cobra.Reaction('UBIQUITIN_MONOMER_DEGRADATIONc')
degradation_ub.subsytem = 'Protein_Expression'
rxn = {seq_amino_acid_map_c[aa_code]: aa_counts for aa_code, aa_counts in monoub_aa_counts.items()}
rxn[ub_c] = -1
rxn[h2o_c] =  -(L_monoub-1)
# atp hydrolysis for translocation/unfolding by 26S - known 1 ATP per 2 residues - https://www.nature.com/articles/s41586-018-0736-4
rxn = hydrolyze_atp(rxn, n_atp = L_monoub/2, compartment = 'c')

degradation_ub.add_metabolites(rxn)
degradation_ub.gene_reaction_rule = ' and '.join(proteasome_machinery)

ub_reactions = ubc_mrna_expression_reactions + [ubc_translation_reaction_cytosolic] 
ub_reactions += ubb_mrna_expression_reactions + [ubb_translation_reaction_cytosolic]
ub_reactions += [ubiquitin_monomerization_ubc, ubiquitin_monomerization_ubb, ubiquitin_monomerization_polyub, degradation_ub]
ub_reactions += [nuclear_import_ub_mono, nuclear_export_ub_poly]

/Users/joycebaghdassarian/opt/anaconda3/envs/human_me/lib/python3.6/site-packages/ipykernel_launcher.py:3 SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/Users/joycebaghdassarian/opt/anaconda3/envs/human_me/lib/python3.6/site-packages/ipykernel_launcher.py:35 SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


# Degradation (Ubiquitin-Proteasome)

In [4]:
def protein_polyubiquitination(gene_info, protein_metabolite, compartment):
    
    polyu_protein_aa_counts = gene_info.amino_acid_counts.copy()
    for aa_code,aa_counts in monoub_aa_counts.items():
        if aa_code in polyu_protein_aa_counts:
            polyu_protein_aa_counts[aa_code] += aa_counts*n_ub
        else: 
            polyu_protein_aa_counts[aa_code] = aa_counts*n_ub
    
    if compartment == 'c':
        if protein_metabolite.compartment != 'c':
            raise ValueError('Compartment mismatch for polyubiquitination')
    
        polyubiquitinate_protein = cobra.Reaction(protein_metabolite.id + 'POLYUBIQUITINATIONc')
        polyubiquitinate_protein.subsytem = 'Protein_Expression'

        polyub_protein_c = make_protein_metabolite(id_ = protein_metabolite.id + '_polyub', 
                           amino_acid_counts = polyu_protein_aa_counts, L_protein = gene_info.L_protein + (L_monoub*n_ub),
                           compartment = 'c') 
        rxn = {protein_metabolite: -1, ub_c: -n_ub, polyub_protein_c:1, h2o_c: n_ub}
        # 1 ATP hydrolysis per ubiquitin monomer added (https://link.springer.com/article/10.1007/s10637-020-00894-6)
        rxn = hydrolyze_atp(rxn, n_atp = n_ub, compartment = 'c')

        polyubiquitinate_protein.add_metabolites(rxn)
        polyubiquitinate_protein.gene_reaction_rule = ' and '.join(UB_ligases_c)
        return polyubiquitinate_protein, polyub_protein_c
    elif compartment == 'n':
        if protein_metabolite.compartment != 'n':
            raise ValueError('Compartment mismatch for polyubiquitination')

        polyubiquitinate_protein = cobra.Reaction(protein_metabolite.id + 'POLYUBIQUITINATIONn')
        polyubiquitinate_protein.subsytem = 'Protein_Expression'

        polyub_protein_n = make_protein_metabolite(id_ = protein_metabolite.id + '_polyub', 
                           amino_acid_counts = polyu_protein_aa_counts, L_protein = gene_info.L_protein + (L_monoub*n_ub),
                           compartment = 'n') 
        rxn = {protein_metabolite: -1, ub_n: -n_ub, polyub_protein_n: 1, h2o_n: n_ub}
        # 1 ATP hydrolysis per ubiquitin monomer added (https://link.springer.com/article/10.1007/s10637-020-00894-6)
        rxn = hydrolyze_atp(rxn, n_atp = n_ub, compartment = 'n')


        polyubiquitinate_protein.add_metabolites(rxn)
        polyubiquitinate_protein.gene_reaction_rule = ' and '.join(UB_ligases_n)
        
        return polyubiquitinate_protein, polyub_protein_n

    else:
        raise ValueError('Current compartment does not have polyubiquitination')

def proteasomal_degradation(gene_info, protein_metabolite, polyub_protein_metabolite, compartment):
    if compartment == 'c':
        if (protein_metabolite.compartment != 'c') or (polyub_protein_metabolite.compartment != 'c'):
            raise ValueError('Compartment mismatch for cytoplasmic proteasomal degradation')
        
        
        deubiquitination = cobra.Reaction(protein_metabolite.id + '_DEUBIQUITINATIONc')
        deubiquitination.subsytem = 'Protein_Expression'
        deubiquitination.add_metabolites({polyub_protein_metabolite: -1, h2o_c: -1, protein_metabolite: 1, polyub_c: 1})
        deubiquitination.gene_reaction_rule = ' and '.join(proteasome_machinery)

        protein_degradation = cobra.Reaction(protein_metabolite.id + '_PROTEASOMAL_DEGRADATIONc')
        protein_degradation.subsytem = 'Protein_Expression'
        rxn = {seq_amino_acid_map_c[aa_code]: aa_counts for aa_code, aa_counts in gene_info.amino_acid_counts.items()}
        rxn[polyub_protein_metabolite], rxn[h2o_c], rxn[polyub_c] = -1, -gene_info.L_protein, 1
        # atp hydrolysis for translocation/unfolding  - known 1 ATP per 2 residues - https://www.nature.com/articles/s41586-018-0736-4
        L_polub_protein = (gene_info.L_protein + (L_monoub*n_ub)) 
        rxn = hydrolyze_atp(rxn, n_atp = L_polub_protein/2, compartment = 'c')


        protein_degradation.add_metabolites(rxn)
        protein_degradation.gene_reaction_rule = ' and '.join(proteasome_machinery)

        protein_degradation_reactions = [deubiquitination, protein_degradation]

        return protein_degradation_reactions
    elif compartment == 'n':
        if (protein_metabolite.compartment != 'n') or (polyub_protein_metabolite.compartment != 'n'):
            raise ValueError('Compartment mismatch for nuclear proteasomal degradation')


        deubiquitination = cobra.Reaction(protein_metabolite.id + '_DEUBIQUITINATIONn')
        deubiquitination.subsytem = 'Protein_Expression'
        deubiquitination.add_metabolites({polyub_protein_metabolite: -1, h2o_n: -1, protein_metabolite: 1, polyub_n: 1})
        deubiquitination.gene_reaction_rule = ' and '.join(proteasome_machinery)

        protein_degradation = cobra.Reaction(protein_metabolite.id + '_PROTEASOMAL_DEGRADATIONn')
        protein_degradation.subsytem = 'Protein_Expression'
        rxn = {seq_amino_acid_map_n[aa_code]: aa_counts for aa_code, aa_counts in gene_info.amino_acid_counts.items()}
        rxn[polyub_protein_metabolite], rxn[h2o_n], rxn[polyub_n] = -1, -gene_info.L_protein, 1
        # atp hydrolysis for translocation/unfolding  - known 1 ATP per 2 residues - https://www.nature.com/articles/s41586-018-0736-4
        L_polub_protein = (gene_info.L_protein + (L_monoub*n_ub)) 
        rxn = hydrolyze_atp(rxn, n_atp = L_polub_protein/2, compartment = 'n')


        protein_degradation.add_metabolites(rxn)
        protein_degradation.gene_reaction_rule = ' and '.join(proteasome_machinery)

        protein_degradation_reactions = [deubiquitination, protein_degradation]

        return protein_degradation_reactions
    else:
        raise ValueError('Current compartment does not have proteasomal degradation')

# Cytosolic Degradation

In [5]:
def degrade_cytosolic_protein(gene_info, folded_protein_c):
    polyubiquitinate_folded_protein_c, polyub_protein_c = protein_polyubiquitination(gene_info, 
                                                          protein_metabolite = folded_protein_c, compartment = 'c') 
    cytosolic_proteasomal_degradation_reactions = proteasomal_degradation(gene_info, 
                                                  protein_metabolite = folded_protein_c, 
                                                  polyub_protein_metabolite = polyub_protein_c, compartment = 'c')
    
    
    return [polyubiquitinate_folded_protein_c] + cytosolic_proteasomal_degradation_reactions

# Nuclear Reactions

In [6]:
def transport_nuclear_protein(gene_info, folded_protein_c):

    folded_protein_n = folded_protein_c.copy()
    folded_protein_n.id, folded_protein_n.compartment = folded_protein_n.id.replace('[c]', '[n]'), 'n' 

    nuclear_import = cobra.Reaction(gene_info.hgnc_id + '_IMPORTtn')
    nuclear_import.subsytem = 'Protein_Expression'
#     nuclear_export = nuclear_import.copy()
#     nuclear_export.id = nuclear_export.id.replace('IMPORT', 'EXPORT')

    import_rxn = {folded_protein_c: -1, folded_protein_n: 1}
#     export_rxn = {folded_protein_c: 1, folded_protein_n: -1}

    if gene_info.protein_mass > nuclear_diffusion_limit:
        # gtp hydrolysis per import
        import_rxn[gtp_n], import_rxn[h2o_n], import_rxn[gdp_n], import_rxn[pi_n], import_rxn[h_n]  = -1, -1, 1, 1, 1
        nuclear_import.add_metabolites(import_rxn)
        nuclear_import.gene_reaction_rule = ' and '.join(importins + RAN)
#         export_rxn[gtp_c], export_rxn[h2o_c], export_rxn[gdp_c], export_rxn[pi_c], export_rxn[h_c]  = -1, -1, 1, 1, 1
#         nuclear_export.add_metabolites(export_rxn)
#         nuclear_export.gene_reaction_rule = ' and '.join(XPO1 + RAN)

    else: # diffusion
        nuclear_import.add_metabolites(import_rxn)
        nuclear_import.lower_bound = -1000
#         nuclear_export.add_metabolites(export_rxn)
        
    return nuclear_import, folded_protein_n

def degrade_nuclear_protein(gene_info, folded_protein_n):
    polyubiquitinate_folded_protein_n, polyub_protein_n = protein_polyubiquitination(gene_info, 
                                                          protein_metabolite = folded_protein_n, compartment = 'n')
    nuclear_proteasomal_degradation_reactions = proteasomal_degradation(gene_info, 
                                                protein_metabolite = folded_protein_n, 
                                                polyub_protein_metabolite = polyub_protein_n, compartment = 'n')
    return [polyubiquitinate_folded_protein_n] + nuclear_proteasomal_degradation_reactions

def get_nuclear_reactions(gene_info, folded_protein_c):
    nuclear_import, folded_protein_n = transport_nuclear_protein(gene_info, folded_protein_c)
    nuclear_degradation_reactions = degrade_nuclear_protein(gene_info, folded_protein_n)
    
    return [nuclear_import] + nuclear_degradation_reactions, folded_protein_n

# Mitochondrial Reactions


In [7]:
# i is intermembrane space, but called inner in compartments BIGG
# stick to notation and use inner instead of inter in reaction naming

def transport_mitochondrial_matrix(gene_info, unfolded_protein_c):
    # transport and folding
    if unfolded_protein_c.compartment != 'c':
        raise ValueError('Only cytoplasmic proteins can be transported to mitochondrial matrix')
    
    mitochondrial_matrix_transport = cobra.Reaction(gene_info.hgnc_id + 'IMPORTtm')
    mitochondrial_matrix_transport.subsytem = 'Protein_Expression'
    pre_protein_m = unfolded_protein_c.copy()
    pre_protein_m.id = pre_protein_m.id.replace('[c]', '[m]')
    pre_protein_m.compartment = 'm'
    pre_protein_m.id = pre_protein_m.id.replace('unfolded', 'folded_pre')
    
    rxn = {unfolded_protein_c: -1, pre_protein_m: 1}
    # ATP hydrolysis for transport, assums 1 ATP consumed per 2 residues
    rxn = hydrolyze_atp(rxn, n_atp = gene_info.L_protein*transport_translocation_atp_cost, compartment = 'm')
    
    
    mitochondrial_matrix_transport.add_metabolites(rxn)
    mitochondrial_matrix_transport.gene_protein_rule = ' and '.join(TOM + TIM23_PAM + HSP70_m)
    
    return mitochondrial_matrix_transport, pre_protein_m

def mitochondrial_matrix_protein_processing(gene_info, pre_protein_m):
    # implement this in the future: cleavage of MTS (and degradation of MTS)
    processed_protein_m, aa_counts_processed_m, L_processed_protein_m = pre_protein_m, gene_info.amino_acid_counts.copy(), gene_info.L_protein
    process_mitochondrial_matrix_protein = None
    return process_mitochondrial_matrix_protein, processed_protein_m, aa_counts_processed_m, L_processed_protein_m


def degrade_mitochondrial_protein(gene_info, protein_metabolite, compartment, L_protein, amino_acid_counts):
    rxn = {seq_amino_acid_map_m[aa_code]: aa_counts for aa_code, aa_counts in amino_acid_counts.items()}
    rxn[protein_metabolite], rxn[h2o_m] = -1, -(L_protein-1)
    
    if compartment == 'm':
        mitochondrial_degradation = cobra.Reaction(gene_info.hgnc_id + '_DEGRADATIONm')
        mitochondrial_degradation.gene_protein_rule = mLON[0]
        
        # ATP hydrolysis by LON: 2 ATP per residue - https://www.ncbi.nlm.nih.gov/pmc/articles/PMC2518814/
        rxn = hydrolyze_atp(rxn, n_atp = L_protein*2, compartment = 'm')
        

    elif compartment == 'i':
        mitochondrial_degradation = cobra.Reaction(gene_info.hgnc_id + '_DEGRADATIONi')
        mitochondrial_degradation.gene_protein_rule = iAAA[0]#' and '.join(mAAA + iAAA)
        
        # ATP hydrolysis by m/i-AAA: 1 ATP per 2 residues -- no source, assumes same as 26S proteasome
        rxn = hydrolyze_atp(rxn, n_atp = L_protein*proteolysis_translocation_atp_cost, compartment = 'i')
  
    mitochondrial_degradation.subsytem = 'Protein_Expression'
    mitochondrial_degradation.add_metabolites(rxn)
    
    return mitochondrial_degradation

def transport_mitochondrial_inter(gene_info, processed_protein_m):
    # upper left Fig 12-29 https://www.ncbi.nlm.nih.gov/books/NBK26828/ 
    # import to matrix then re-export to inter membrane space
    
    if processed_protein_m.compartment != 'm':
        raise ValueError('Only the mechanism of mitochondrial matrix import and re-export to inter membrane is considered')
    
    mitochondrial_inter_transport = cobra.Reaction(gene_info.hgnc_id + 'IMPORTti')
    mitochondrial_inter_transport.subsytem = 'Protein_Expression'
    pre_protein_i = processed_protein_m.copy()
    pre_protein_i.id = processed_protein_m.id.replace('[m]', '[i]')
    pre_protein_i.compartment = 'i'
    
    rxn = {processed_protein_m: -1, pre_protein_i: 1}
    
    mitochondrial_inter_transport.add_metabolites(rxn)
    mitochondrial_inter_transport.gene_protein_rule = OXA[0]
    
    return mitochondrial_inter_transport, pre_protein_i

def mitochondrial_inter_protein_processing(gene_info, pre_protein_i):
    # implement this in the future: cleavage of secondary sequence (and degradation)
    processed_protein_i, aa_counts_processed_i, L_processed_protein_i = pre_protein_i, gene_info.amino_acid_counts.copy(), gene_info.L_protein
    process_mitochondrial_matrix_protein = None
    return process_mitochondrial_matrix_protein, processed_protein_i, aa_counts_processed_i, L_processed_protein_i

def get_mitochondrial_reactions(gene_info, unfolded_protein_c, compartments):
    mitochondrial_matrix_transport, pre_protein_m = transport_mitochondrial_matrix(gene_info, unfolded_protein_c)
    process_mitochondrial_matrix_protein, processed_protein_m, aa_counts_processed_m, L_processed_protein_m = mitochondrial_matrix_protein_processing(gene_info, pre_protein_m)
    
    mitochondrial_reactions = [mitochondrial_matrix_transport]
    mitochondrial_protein_metabolites = list()
    if process_mitochondrial_matrix_protein != None:
        mitochondrial_reactions += [process_mitochondrial_matrix_protein]
    
    if 'm' in compartments:
        mitochondrial_matrix_degradation = degrade_mitochondrial_protein(gene_info, protein_metabolite = processed_protein_m, compartment = 'm', L_protein = L_processed_protein_m, amino_acid_counts = aa_counts_processed_m)
        mitochondrial_reactions += [mitochondrial_matrix_degradation]
        mitochondrial_protein_metabolites += [processed_protein_m]
    if 'i' in compartments:
        mitochondrial_inter_transport, pre_protein_i = transport_mitochondrial_inter(gene_info, processed_protein_m)
        process_mitochondrial_inter_protein, processed_protein_i, aa_counts_processed_i, L_processed_protein_i = mitochondrial_inter_protein_processing(gene_info, pre_protein_i)
        if process_mitochondrial_matrix_protein != None:
                mitochondrial_reactions += [process_mitochondrial_inter_protein]        
        mitochondrial_inter_degradation = degrade_mitochondrial_protein(gene_info, protein_metabolite = processed_protein_i, compartment = 'i', L_protein = L_processed_protein_i, amino_acid_counts = aa_counts_processed_i)
        mitochondrial_reactions += [mitochondrial_inter_transport, mitochondrial_inter_degradation]
        mitochondrial_protein_metabolites += [processed_protein_i]

    return mitochondrial_reactions, mitochondrial_protein_metabolites

# Peroxisomal

In [8]:
def transport_peroxisome(gene_info, folded_protein_c):
    if folded_protein_c.compartment != 'c':
        raise ValueError('Only cytoplasmic proteins can be transported to mitochondrial matrix')
    
    peroxisomal_transport = cobra.Reaction(gene_info.hgnc_id + 'IMPORTtx')
    peroxisomal_transport.subsytem = 'Protein_Expression'
    folded_protein_x = folded_protein_c.copy()
    folded_protein_x.id = folded_protein_x.id.replace('[c]', '[x]')
    folded_protein_x.compartment = 'x'
    
    rxn = {folded_protein_c: -1, folded_protein_x: 1}
    # ATP hydrolysis for transport--translocation of protein, export of PEX5S receptor
    rxn = hydrolyze_atp(rxn, n_atp = (gene_info.L_protein+L_PEX5)*transport_translocation_atp_cost, 
                        compartment = 'x')
    
    
    peroxisomal_transport.add_metabolites(rxn)
    peroxisomal_transport.gene_protein_rule = ' and '.join(peroxins + AWP1)
    
    return peroxisomal_transport, folded_protein_x

def degrade_peroxisomal_protein(gene_info, folded_protein_x):
    
    
    peroxisomal_degradation = cobra.Reaction(gene_info.hgnc_id + 'DEGRADATIONx')
    peroxisomal_degradation.subsytem = 'Protein_Expression'
    peroxisomal_degradation.gene_protein_rule = LONP2[0]

    rxn = {seq_amino_acid_map_x[aa_code]: aa_counts for aa_code, aa_counts in gene_info.amino_acid_counts.items()}
    rxn[folded_protein_x], rxn[h2o_x] = -1, -(gene_info.L_protein-1)
    # ATP hydrolysis by LON: 2 ATP per residue - https://www.ncbi.nlm.nih.gov/pmc/articles/PMC2518814/
    rxn = hydrolyze_atp(rxn, n_atp = gene_info.L_protein*2, compartment = 'x')
    peroxisomal_degradation.add_metabolites(rxn)
    
    return peroxisomal_degradation

def get_peroxisomal_reactions(gene_info, folded_protein_c):
    peroxisomal_transport, folded_protein_x = transport_peroxisome(gene_info, folded_protein_c)
    peroxisomal_degradation = degrade_peroxisomal_protein(gene_info, folded_protein_x)
    
    return [peroxisomal_transport, peroxisomal_degradation], folded_protein_x

# Secretory Pathway

adapted from Jahir's Recon2_2s

# ER transport

In [99]:
def post_translational_translocation(gene_info, unfolded_protein_c):
    if gene_info.L_protein > 160:
        raise ValueError('This protein is too long for post-translational translocation')
    if unfolded_protein_c.compartment != 'c':
        raise ValueError('Protein metabolite is not in cytosolic compartment')
    
    ptt_reactions = list()
    
    folded_protein_r = unfolded_protein_c.copy()
    folded_protein_r.id = folded_protein_r.id.replace('[c]', '[r]')
    folded_protein_r.id = folded_proetein_r.id.replace('unfolded', 'folded')
    folded_protein_r.compartment = 'r'
    rxn = {unfolded_protein_c: -1, folded_protein_r: 1}
    rxn = hydrolyze_atp(rxn, n_atp = 1, compartment = 'c')
    
    
    if gene_info.tmd > 0 or 'pm' in gene_info.final_locations.keys(): # membrane secreted protein
        post_translational_translocation_r = cobra.Reaction(gene_info.hgnc_id + '_post_TRANSLOC_3A_IMPORTtr')
        post_translational_translocation_r.subsytem = 'Protein_Expression'
        post_translational_translocation_r.gene_reaction_rule = ' and '.join(ASNA1 + WRB+translation_efs + ['ribosome'])
        
        # complex cleavage from jahir's (+h2o_c, +h_c, + pi_c) not included 
        post_translational_translocation_r.add_metabolites(rxn)
        ptt_reactions += [post_translational_translocation_r]
     
    c_ = [c for c, t in gene_info.final_locations.items() if t == 'Canonical Secretion']
    if gene_info.tmd == 0 and len(set(c_).difference(['pm'])) > 1: #non membrane secreted protein
        number_BiP = gene_info.L_protein/40
        post_translational_translocation_r = cobra.Reaction(gene_info.hgnc_id + '_post_TRANSLOC_3B_IMPORTtr')
        post_translational_translocation_r.subsytem = 'Protein_Expression'
        post_translational_translocation_r.gene_reaction_rule = ' and '.join(ptnm+translation_efs + ['ribosome'])
        
        rxn = hydrolyze_atp(rxn, n_atp = number_BiP, compartment = 'r')
        post_translational_translocation_r.add_metabolites(rxn)
        ptt_reactions += [post_translational_translocation_r]
    
    

    return ptt_reactions, folded_protein_r

def co_translational_translocation(gene_info):
    if gene_info.L_protein <= 160:
        raise ValueError('This protein is too short for co-translational translocation')    
    
    ctt_reactions = list()
    
    # reaction metabolites------------------------------------------------------------------------------------
    number_BiP = gene_info.L_protein/40
    
    rxn = {charged_trna_map[aa_code]: -aa_count for aa_code, aa_count in gene_info.amino_acid_counts.items()} # tRNA consumption
    rxn[modified_trna_transcript_c] = gene_info.L_protein
    rxn[h2o_c] = -gene_info.L_protein # release of peptide from tRNA, addition of -OH to uncharged tRNA
    rxn[h_c] = gene_info.L_protein # release of peptide from tRNA, addition of -OH to uncharged tRNA
    rxn[h2o_c] += gene_info.L_protein - 1 # peptide bond formation (hydrolysis)
    
    # gtp hydrolysis per aa added
    rxn[ntp_map_c['G']] = -gene_info.L_protein 
    rxn[h2o_c] -= gene_info.L_protein
    rxn[ndp_map_c['G']] = gene_info.L_protein
    rxn[pi_c] = gene_info.L_protein
    rxn[h_c] += gene_info.L_protein
    unprocessed_protein_r = make_protein_metabolite(id_ = gene_info.hgnc_id + '_unprocessed_folded', 
                amino_acid_counts = gene_info.amino_acid_counts, L_protein = gene_info.L_protein,
                compartment = 'r')

    rxn[unprocessed_protein_r] = 1
    rxn = hydrolyze_atp(rxn, n_atp = number_BiP, compartment = 'r')
    #------------------------------------------------------------------------------------

    co_translational_translocation_r = cobra.Reaction(gene_info.hgnc_id + '_co_TRANSLOC_IMPORTtr')
    co_translational_translocation_r.subsytem = 'Protein_Expression'
    co_translational_translocation_r.add_metabolites(rxn)
    co_translational_translocation_r.gene_reaction_rule = ' and '.join(ctnm + translation_efs + ['ribosome'])
    ctt_reactions += [co_translational_translocation_r]
    
    # sp degradation
    sp_seq = gene_info.protein_seq[:L_sp]
    sp_aa_counts = {k: sp_seq.count(k) for k in amino_acids}
    gene_info.protein_seq = gene_info.protein_seq[L_sp:]
    gene_info.amino_acid_counts = {k: gene_info.protein_seq.count(k) for k in amino_acids}
    gene_info.L_protein = len(gene_info.protein_seq)
    gene_info.protein_mass = calculate_molecular_weight(seq=gene_info.protein_seq, seq_type='protein')

    folded_protein_r = make_protein_metabolite(id_ = gene_info.hgnc_id + '_folded', 
            amino_acid_counts = gene_info.amino_acid_counts, L_protein = gene_info.L_protein,
            compartment = 'r')

    rxn = {seq_amino_acid_map_r[aa]: count for aa, count in sp_aa_counts.items()}
    rxn[h2o_r] = -L_sp
    rxn[unprocessed_protein_r],rxn[folded_protein_r] = -1, 1
    
    sp_degradation = cobra.Reaction(gene_info.hgnc_id + '_SP_degradation')
    sp_degradation.subsystem = 'Protein Expression'
    sp_degradation.add_metabolites(rxn)
    sp_degradation.gene_reaction_rule = sp_rule
    ctt_reactions += [sp_degradation]

    return ctt_reactions, folded_protein_r, gene_info

# ER Modifications

In [435]:
def form_disulfide_bond(gene_info, folded_protein_r):
    number_DSB = gene_info.ptms['dsb']
    disulfide_bond_formation = cobra.Reaction(gene_info.hgnc_id + '_DSBr')
    disulfide_bond_formation.subsystem = 'Protein Expression'
    modified_protein_dsb_r = folded_protein_r.copy()
    modified_protein_dsb_r.id = modified_protein_dsb_r.id.replace('folded', 'folded_DSB')
    elements = folded_protein_r.elements.copy()
    elements['H'] -= 2*number_DSB
    modified_protein_dsb_r.elements = elements
    # diagram https://www.google.com/url?sa=i&url=https%3A%2F%2Fen.wikipedia.org%2Fwiki%2FProtein_disulfide-isomerase&psig=AOvVaw0bGpff4XX1eYEF61H1RJKw&ust=1597273135069000&source=images&cd=vfe&ved=0CAIQjRxqFwoTCJi6l6GglOsCFQAAAAAdAAAAABAJ
    rxn = {folded_protein_r: -1, modified_protein_dsb_r: 1, o2_r: -number_DSB, h2o2_r: number_DSB}
    disulfide_bond_formation.add_metabolites(rxn)
    disulfide_bond_formation.gene_reaction_rule = P4HB[0]
    
    return disulfide_bond_formation, modified_protein_dsb_r

def form_gpi(gene_info, modified_protein_r):
    gpi_formation = cobra.Reaction(gene_info.hgnc_id + '_GPIr')
    gpi_formation.subsystem = 'Protein Expression'
    modified_protein_gpi_r = modified_protein_r.copy()
    modified_protein_gpi_r.id = modified_protein_gpi_r.id.replace('folded', 'folded_GPI')

    elements = modified_protein_r.elements.copy()
    for e,c in balanced_gpi.items():
        if e in elements.keys():
            elements[e] += c
        else:
            elements[e] = c
    modified_protein_gpi_r.elements = elements

    rxn = M4ATAer.copy() # need these additional metabolties to get mass balance with gpi_sig[r]
    rxn[hdca_r], rxn[gpi_hs_r], rxn[h_r], rxn[h2o_r] = 1,-1,1,-1
    rxn[modified_protein_r], rxn[modified_protein_gpi_r]= -1, 1
    gpi_formation.add_metabolites(rxn)
    gpi_formation.gene_reaction_rule = ' and '.join(gpi_machinery)
    
    return gpi_formation, modified_protein_gpi_r

def glycosylate_n_linked(gene_info, modified_protein_r):
    raise ValueError('N-glycosylation not yet incorporated')
#     n_glycosylation = cobra.Reaction(gene_info.hgnc_id + 'NGLYCOr')
#     n_glycosylation.subsystem = 'Protein Expression'
#     modified_protein_ng_r = modified_protein_r.copy()
#     modified_protein_ng_r.id = modified_protein_ng_r.id.replace('folded', 'folded_NG')

    # # add metabolites and GPRS
    # return n_glycosylation, modified_protein_ng_r
    


def modify_protein_er(gene_info, folded_protein_r):
    if folded_protein_r.compartment != 'r':
        raise ValueError('Only er compartment proteins can get disulfide bonds, GPI anchors, or n glycosylation')
    
    
    modification_reactions = list()
    
    if 'dsb' in gene_info.ptms.keys() and gene_info.ptms['dsb'] > 0:
        disulfide_bond_formation, modified_protein_r = form_disulfide_bond(gene_info, folded_protein_r)
        modification_reactions += [disulfide_bond_formation]
    else:
        modified_protein_r = folded_protein_r # these lines update the protein metabolite to be appropriate inputs to proceeding functions

    if 'gpi' in gene_info.ptms.keys() and gene_info.ptms['gpi'] > 0: # == 1 but doesn't matter bc of gene_info checks
        
        gpi_formation, modified_protein_r = form_gpi(gene_info, modified_protein_r)
        modification_reactions += [gpi_formation]
    else:
        modified_protein_r = modified_protein_r
    
    # N GLYCOSYLATION here must be updated   
    if 'ng' in gene_info.ptms.keys() and gene_info.ptms['ng'] > 0: 
        n_glycosylation, modified_protein_r = glycosylate_n_linked(gene_info, modified_protein_r)
        modification_reactions += [n_glycosylation]
    else:
        modified_protein_r = modified_protein_r
    
    
    return modification_reactions, modified_protein_r

# Golgi Reactions

In [554]:
# add to utils
Kv = 0.7

copii_r_m = ['HGNC:14562', 'HGNC:4430', 'HGNC:6632', 'HGNC:9758', 'HGNC:10535', 'HGNC:10697', 'HGNC:29006', 
             'HGNC:10700', 'HGNC:10701', 'HGNC:10703', 'HGNC:17052', 'HGNC:11440']
copii_gpi_m = ['HGNC:14562', 'HGNC:4430', 'HGNC:9758', 'HGNC:10535', 'HGNC:10697', 'HGNC:29006', 'HGNC:10700', 
               'HGNC:10701', 'HGNC:10703', 'HGNC:17052', 'HGNC:11440']
udpacgal_g = human_model.metabolites.get_by_id('udpacgal[g]')
udpgal_g = human_model.metabolites.get_by_id('udpgal[g]')
uacgam_g = human_model.metabolites.get_by_id('uacgam[g]')
h_g = human_model.metabolites.get_by_id('h[g]')
udp_g = human_model.metabolites.get_by_id('udp[g]')

og_rule = '(HGNC:16347 or HGNC:19873 or HGNC:4124 or HGNC:4127 or HGNC:4131 or HGNC:4125 or HGNC:4129 or HGNC:19875 or HGNC:4128 or HGNC:23242 or HGNC:4123 or HGNC:4130 or HGNC:4126) and HGNC:24337 and HGNC:24338 and HGNC:4205'
copi_m = ['HGNC:649', 'HGNC:14562', 'HGNC:2230', 'HGNC:2231', 'HGNC:2232', 'HGNC:2234', 'HGNC:2236', 'HGNC:2243', 'HGNC:19356', 
          'HGNC:9758', 'HGNC:10700', 'HGNC:11443', 'HGNC:15942', 'HGNC:25847']
clathrin_m = ['HGNC:652', 'HGNC:2090', 'HGNC:2091', 'HGNC:2092', 'HGNC:17842', 'HGNC:16064', 'HGNC:17079', 
              'HGNC:14902', 'HGNC:11441', 'HGNC:11442', 'HGNC:11430']

# clathrin_coeff = int(round(29880.01 * Kv / V)) # Number of proteins per clathrin vesicle  
# copi_coeff = int(round(143793.19 * Kv / V))


In [555]:
def import_golgi(gene_info, modified_protein_r):
    V = gene_info.protein_mass * 1.21 / 1000.0 # Protein Volume in nm^3
    copii_coeff = int(round(268082.35 * Kv / V))
    
    protein_g = modified_protein_r.copy()
    protein_g.id = protein_g.id.replace('[r]', '[g]')
    protein_g.compartment = 'g'
    
    rxn = {modified_protein_r: -copii_coeff, protein_g: copii_coeff}
    rxn[ntp_map_c['G']], rxn[h2o_c], rxn[ndp_map_c['G']], rxn[pi_c], rxn[h_c]  = -94, -94, 94, 94, 94

    golgi_import = cobra.Reaction(gene_info.hgnc_id + '_COPII_IMPORTtg')
    golgi_import.subsystem = 'Protein Expression'
    golgi_import.add_metabolites(rxn)
    
    
    if 'gpi' in gene_info.ptms.keys() and 'ng' not in gene_info.ptms.keys(): # this if statement is analogous to Recon2.2S's connector statements in copii reactions
        golgi_import.gene_reaction_rule = ' and '.join(copii_gpi_m)
    else:
        golgi_import.gene_reaction_rule = ' and '.join(copii_r_m)
        
    
    return golgi_import, protein_g

def glycosylate_o_linked(gene_info, protein_g):
    number_Oglycans = gene_info.ptms['og']
    o_glycosylation = cobra.Reaction(gene_info.hgnc_id + '_OGg')
    o_glycosylation.subsystem = 'Protein Expression'

    # metabolites
    modified_protein_og_g = protein_g.copy()
    modified_protein_og_g.id = modified_protein_og_g.id.replace('folded', 'folded_OG')

    balance_og = {'C': (8 + 6 + 8)*number_Oglycans, # each 1/3 entry is for each 1/3 reactions in Jahir's model, in case want to separate in the future 
                  'H': (13 + 10 + 13)*number_Oglycans, 
                  'N': (1 + 0 + 1)*number_Oglycans, 
                  'O': (5 + 5 + 5)*number_Oglycans}
    elements = modified_protein_og_g.elements.copy()
    for e,c in balance_og.items():
        if e in elements.keys():
            elements[e] += c
        else:
            elements[e] = c
    modified_protein_og_g.elements = elements

    rxn = {protein_g: -1, modified_protein_og_g: 1, udpacgal_g: -number_Oglycans, 
          udpgal_g: -number_Oglycans, uacgam_g: -number_Oglycans, h_g: 3*number_Oglycans, udp_g: 3* number_Oglycans}
    o_glycosylation.add_metabolites(rxn)
    
    o_glycosylation.gene_reaction_rule = og_rule
    
    return o_glycosylation, modified_protein_og_g
    

def modify_protein_golgi(gene_info, protein_g):
    if protein_g.compartment != 'g':
        raise ValueError('Only golgi compartment proteins can be O-glycosylated')
    
    # this set up allows for incorporation of other Golgi PTMs in the future, similar to modify_protein_er fct
    modification_reactions = list()
    if 'og' in gene_info.ptms.keys() and gene_info.ptms['og'] > 0:
        o_glycosylation, modified_protein_g = glycosylate_o_linked(gene_info, protein_g)
        modification_reactions += [o_glycosylation]
    else:
        modified_protein_g = protein_g
    
    
    return modification_reactions, modified_protein_g


def retrograde_er(gene_info, modified_protein_g):
    V = gene_info.protein_mass * 1.21 / 1000.0 # Protein Volume in nm^3
    copi_coeff = int(round(143793.19 * Kv / V))

    retro_protein_r = modified_protein_g.copy()
    retro_protein_r.id = retro_protein_r.id.replace('[g]', '[r]')
    retro_protein_r.compartment = 'r'

    rxn = {modified_protein_g: -copi_coeff, retro_protein_r: copi_coeff}
    rxn[ntp_map_c['G']], rxn[h2o_c], rxn[ndp_map_c['G']], rxn[pi_c], rxn[h_c]  = -127, -127, 127, 127, 127

    retrograde_transport = cobra.Reaction(gene_info.hgnc_id + '_COPI_RETROtr')
    retrograde_transport.subsystem = 'Protein Expression'
    retrograde_transport.add_metabolites(rxn)
    retrograde_transport.gene_reaction_rule = ' and '.join(copi_m)
    
    return retrograde_transport, retro_protein_r

# Lysosomal, Extracellular, and Plasma Membrane Transport

In [574]:
def secrete_protein(gene_info, modified_protein_g):
    V = gene_info.protein_mass * 1.21 / 1000.0 # Protein Volume in nm^3
    clathrin_coeff = int(round(29880.01 * Kv / V)) # Number of proteins per clathrin vesicle  

    secreted_proteins = list()
    if 'e' in gene_info.final_locations.keys():
        secreted_protein = modified_protein_g.copy()
        secreted_protein.id = secreted_protein.id.replace('[g]', '[e]')
        secreted_protein.compartment = 'e'
        secreted_proteins += [secreted_protein]
    if 'pm' in gene_info.final_locations.keys():
        secreted_protein = modified_protein_g.copy()
        secreted_protein.id = secreted_protein.id.replace('[g]', '[pm]')
        secreted_protein.compartment = 'pm'
        secreted_proteins += [secreted_protein]
    if 'l' in gene_info.final_locations.keys():
        secreted_protein = modified_protein_g.copy()
        secreted_protein.id = secreted_protein.id.replace('[g]', '[l]')
        secreted_protein.compartment = 'l'
        secreted_proteins += [secreted_protein]

    secreted_protein_reactions = list()
    for secreted_protein in secreted_proteins:

        rxn = {modified_protein_g: -clathrin_coeff, secreted_protein: clathrin_coeff}
        rxn[ntp_map_c['G']], rxn[h2o_c], rxn[ndp_map_c['G']], rxn[pi_c], rxn[h_c]  = -44, -44, 44, 44, 44

        secrete_protein = cobra.Reaction(gene_info.hgnc_id + '_Clathrin_IMPORTt' + secreted_protein.compartment)
        secrete_protein.subsystem = 'Protein Expression'
        secrete_protein.add_metabolites(rxn)
        secrete_protein.gene_reaction_rule = ' and '.join(clathrin_m)
        secreted_protein_reactions += [secrete_protein]

    return secreted_protein_reactions, secreted_proteins



# Secretory Pathway Protein Degradation

In [556]:
# Jahir's NCBI GPRs to HGNC GPRs
import re
ehm = pd.read_csv(local_data_path + 'raw/identifiers.txt', sep = '\t')
ehm = ehm.loc[ehm['NCBI gene ID'].dropna().index,:]
ehm['NCBI gene ID'] = ehm['NCBI gene ID'].astype('int64').astype(str)


test = ['',
 '(1211) and (1212) and (1213) and (26088) and (23062) and (23163) and (51560) and (8417) and (375)',
 '(1211) and (1212) and (1213) and (26088) and (23062) and (23163) and (51560) and (8417) and (375) and (10228) and (23673)',
 '',
 '']
test = [re.findall(r'\d+', i) for i in test]
test = sorted(set([item for sublist in test for item in sublist]))
L_test = len(test)
ehm = ehm[ehm['NCBI gene ID'].isin(test)]

if len(ehm['NCBI gene ID'].unique()) != L_test:
    print(set(test).difference(ehm['NCBI gene ID'].tolist()))
if len(ehm['NCBI gene ID'].unique()) != ehm.shape[0]:
    print('Redundant genes')
    
mapper = dict(zip(ehm['NCBI gene ID'], ehm['HGNC ID']))

In [558]:
print(ehm['HGNC ID'].tolist())

['HGNC:652', 'HGNC:2090', 'HGNC:2091', 'HGNC:2092', 'HGNC:17842', 'HGNC:16064', 'HGNC:17079', 'HGNC:14902', 'HGNC:11441', 'HGNC:11442', 'HGNC:11430']


# Protein Expression All

In [575]:
def get_protein_expression_reactions(gene_info):
    # after transport, expand these to secretory pathways
    protein_expression_reactions, protein_metabolites = list(), list()
    
    # transport: c, n, m, i, x and post-translational translocation
    if 'Cytosolic Tranport' in gene_info.final_locations.values() or gene_info.L_protein <= 160: 
        translation_elongation_c, unfolded_protein_c = translate_protein_cytosolic(gene_info)
        protein_expression_reactions.append(translation_elongation_c)
        
        if 'Cytosolic Tranport' in gene_info.final_locations.values():
            if 'c' in gene_info.final_locations.keys() or 'x' in gene_info.final_locations.keys() or 'n' in gene_info.final_locations.keys():
                protein_folding_cytosolic, folded_protein_c = fold_protein_cytosolic(gene_info, unfolded_protein_c)
                protein_expression_reactions += [protein_folding_cytosolic]


                if 'c' in gene_info.final_locations.keys() or 'x' in gene_info.final_locations.keys() or ('n' in gene_info.final_locations.keys() and gene_info.protein_mass <= nuclear_diffusion_limit):
                   # cytoplasmic degradation of folded proteins: cytoplasmic proteins, peroxisomal proteins, or nuclear proteins undergoing passive diffusion
                    protein_expression_reactions += degrade_cytosolic_protein(gene_info, folded_protein_c)

                    if 'c' in gene_info.final_locations.keys():
                        protein_metabolites += [folded_protein_c]

                    if 'x' in gene_info.final_locations.keys():
                        peroxisomal_reactions, folded_protein_x = get_peroxisomal_reactions(gene_info, folded_protein_c)
                        protein_expression_reactions += peroxisomal_reactions
                        protein_metabolites += [folded_protein_x]

                if 'n' in gene_info.final_locations.keys():
                    nuclear_reactions, folded_protein_n = get_nuclear_reactions(gene_info, folded_protein_c)
                    protein_expression_reactions += nuclear_reactions
                    protein_metabolites += [folded_protein_n]


            if 'i' in gene_info.final_locations.keys(): # no folding for i, but cytoplasmic degradation
                protein_expression_reactions += degrade_cytosolic_protein(gene_info, unfolded_protein_c)

            # mitochondrial transport and degradation ('i' and 'm')
            if ('m' in gene_info.final_locations.keys()) or ('i' in gene_info.final_locations.keys()):
                if ('m' in gene_info.final_locations.keys()) and ('i' in gene_info.final_locations.keys()):
                    mitochondrial_reactions, mitochondrial_protein_metabolites = get_mitochondrial_reactions(gene_info, unfolded_protein_c, compartments = ['m','i'])
                elif 'm' in gene_info.final_locations.keys():
                    mitochondrial_reactions, mitochondrial_protein_metabolites = get_mitochondrial_reactions(gene_info, unfolded_protein_c, compartments = ['m'])
                elif 'i' in gene_info.final_locations.keys():
                    mitochondrial_reactions, mitochondrial_protein_metabolites = get_mitochondrial_reactions(gene_info, unfolded_protein_c, compartments = ['i'])
                protein_expression_reactions += mitochondrial_reactions
                protein_metabolites += mitochondrial_protein_metabolites
    
    # SECRETORY PATHWAY            
    if 'Canonical Secretion' in gene_info.final_locations.values():
        if gene_info.L_protein <= 160: # post translational translocation
            ptt_reactions, folded_protein_r = post_translational_translocation(gene_info, unfolded_protein_c)
            protein_expression_reactions += ptt_reactions
        else: # co translational translocation
            ctt_reactions, folded_protein_r, gene_info = co_translational_translocation(gene_info)
            protein_expression_reactions += ctt_reactions
        
        # er ptms
        if 'dsb' in gene_info.ptms.keys() or 'gpi' in gene_info.ptms.keys() or 'ng' in gene_info.ptms.keys():
            modification_er_reactions, modified_protein_r = modify_protein_er(gene_info, folded_protein_r)
            protein_expression_reactions += modification_er_reactions
        else:
            modified_protein_r = folded_protein_r
        
        
        if len(set(['g', 'pm', 'e', 'l']).intersection(gene_info.final_locations.keys())) > 0 or 'og' in gene_info.ptms.keys():
            golgi_import, protein_g = import_golgi(gene_info, modified_protein_r)
            protein_expression_reactions += [golgi_import]
            
            # golgi ptms
            if 'og' in gene_info.ptms.keys():
                modification_golgi_reactions, modified_protein_g = modify_protein_golgi(gene_info, protein_g)
                protein_expression_reactions += modification_golgi_reactions
            else: 
                modified_protein_g = protein_g
                
            # transport to plasma membrane, extracellular, and lysosome    
            if len(set(['pm', 'e', 'l']).intersection(gene_info.final_locations.keys())) > 0:
                secreted_protein_reactions, secreted_proteins = secrete_protein(gene_info, modified_protein_g)
                protein_expression_reactions += secreted_protein_reactions
                protein_metabolites += secreted_proteins
                            
            
            # retrograde transport
            if 'g' in gene_info.final_locations.keys() or 'r' in gene_info.final_locations.keys():
                # golgi retrograde transport for degradation
                retrograde_transport, retro_protein_r = retrograde_er(gene_info, modified_protein_g)
                protein_expression_reactions += [retrograde_transport]
                if 'g' in gene_info.final_locations.keys():
                    protein_metabolites += [modified_protein_g]

        else:
            retro_protein_r = modified_protein_r # for ER resident proteins with no O-glycosylation, they are not transported to Golgi and retrograde transported
        
        if 'r' in gene_info.final_locations.keys():
            protein_metabolites += [retro_protein_r]
        # ER degradation and PM/L degradation needed

    elif 'Non-Canonical Secretion' in gene_info.final_locations.values():
        raise ValueError('Model does not currently account for non-canonical secretion')
    
        
    return protein_expression_reactions, protein_metabolites

In [576]:
gene_info = generate_geneinfo_object(hgnc_id = 'HGNC:549', final_locations = [], 
                                     psim = psim_me, keff = None, n_introns = None)


In [582]:
gene_info.final_locations = {'l': 'Canonical Secretion', 'g': 'Canonical Secretion'}
gene_info.ptms = {'gpi': 1, 'og': 5}
protein_expression_reactions, protein_metabolites = get_protein_expression_reactions(gene_info)

In [583]:
protein_expression_reactions

[<Reaction HGNC:549_co_TRANSLOC_IMPORTtr at 0x7f88b4efe0b8>,
 <Reaction HGNC:549_SP_degradation at 0x7f88bc3b6ba8>,
 <Reaction HGNC:549_GPIr at 0x7f88bc3b6208>,
 <Reaction HGNC:549_COPII_IMPORTtg at 0x7f88bc3b6b38>,
 <Reaction HGNC:549_OGg at 0x7f88bc3b6dd8>,
 <Reaction HGNC:549_Clathrin_IMPORTtl at 0x7f88ba211fd0>,
 <Reaction HGNC:549_COPI_RETROtr at 0x7f88ba211eb8>]

In [498]:
[r.check_mass_balance() for r in protein_expression_reactions]

[{}, {}, {}, {}, {}]

In [569]:
human_model.compartments

{'c': '',
 'e': '',
 'l': '',
 'm': '',
 'r': '',
 'n': '',
 'g': '',
 'x': '',
 'i': ''}

In [ ]:
#ub_reactions